In [1]:
%matplotlib widget
%matplotlib widget
import os
from pathlib import Path
import time
import torch
import numpy as np
import math
import gc
from functools import partial
from dataset_alt import Dataset, load_dataframes_from_folder, reverse_normalization
from torch.utils.data import DataLoader
from transformer_zerostep import GPTConfig, GPT, warmup_cosine_lr, GPT_chopped
import argparse
import warnings
import matplotlib.pyplot as plt
# import onnxruntime as rt
# import onnx
import copy
from collections import OrderedDict
import pickle as pkl


# set figure parameters
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42
plt.rcParams["font.family"] = "Times New Roman"
plt.rcParams["mathtext.fontset"] = "cm"
plt.rcParams['axes.labelsize']=14
plt.rcParams['xtick.labelsize']=11
plt.rcParams['ytick.labelsize']=11
plt.rcParams['axes.grid']=True
plt.rcParams['axes.xmargin']=0

RuntimeError: 'widget' is not a recognised GUI loop or backend name

In [19]:
# Overall settings
out_dir = "out"

model_name = "noise_h50_40k.pt"
# model_name = "model_high_speed.pt"

# current_path = os.getcwd().split("in-context-bldc")[0]
# data_path = os.path.join(current_path,"in-context-bldc", "data")

# folder = "simulated/50_percent_control/validation"
# # folder = "CL_experiments_double_sensor_control/test/inertia13"
# folder_path = os.path.join(data_path, folder)

# Compute settings
cuda_device = "cuda:0"
no_cuda = True
threads = 10
compile = False

# Configure compute
torch.set_num_threads(threads) 
use_cuda = not no_cuda and torch.cuda.is_available()
device_name  = cuda_device if use_cuda else "cpu"
device = torch.device(device_name)
device_type = 'cuda' if 'cuda' in device_name else 'cpu' # for later use in torch.autocast
torch.set_float32_matmul_precision("high")
print(torch.cuda.is_available())
# Create out dir
out_dir = Path(out_dir)
exp_data = torch.load(out_dir/model_name, map_location=device, weights_only=False)
seq_len = exp_data["cfg"].seq_len
nx = exp_data["cfg"].nx
exp_data["iter_num"]
print(seq_len)
print(exp_data["iter_num"])
print(exp_data['best_val_loss'])
print(exp_data["cfg"])
print(exp_data["cfg"].lr)
print(exp_data["train_time"]/3600)
print(exp_data["model_args"])

False
50
37331
0.0011266251094639301
Namespace(model_dir='out', out_file='noise_h50', in_file='noise_h50_10k', init_from='scratch', seed=42, log_wandb=False, nx=4, nu=6, ny=1, seq_len=50, mag_range=(0.5, 0.97), phase_range=(0.0, 1.5707963267948966), fixed_system=False, n_layer=8, n_head=4, n_embd=16, dropout=0, bias=False, batch_size=128, max_iters=40000, warmup_iters=5000, lr=1e-05, weight_decay=0.0, eval_interval=10, eval_iters=10, fixed_lr=False, threads=16, no_cuda=False, cuda_device='cuda:0', compile=False, beta1=0.9, beta2=0.95, block_size=50, lr_decay_iters=40000, min_lr=1.0000000000000002e-06, decay_lr=True, eval_batch_size=128)
1e-05
15.143221220042971
{'n_layer': 8, 'n_head': 4, 'n_embd': 16, 'n_x': 4, 'n_y': 1, 'n_u': 6, 'block_size': 50, 'bias': False, 'dropout': 0}


In [20]:
# generate the model
model_args = exp_data["model_args"]
gptconf = GPTConfig(**model_args)
model = GPT(gptconf).to(device)


state_dict = exp_data["model"]

keys_raw = state_dict.keys()
unwanted_prefix = '_orig_mod.'
for k,v in list(state_dict.items()):
    if k.startswith(unwanted_prefix):
        state_dict[k[len(unwanted_prefix):]] = state_dict.pop(k)
    if k.startswith('module.'):
        state_dict[k[7:]] = v
        state_dict.pop(k)

keys = state_dict.keys()


model.load_state_dict(state_dict)

number of parameters: 0.03M


<All keys matched successfully>

In [21]:
print(len(keys_raw))
print(len(keys))

54
54


In [22]:
# model_name[:-3]

In [23]:
test_input = torch.rand(5,gptconf.block_size,gptconf.n_u)
print(test_input.size())
test_output = model(test_input)
print(test_output.size())
# print(test_output)
gptconf_ord_dict = OrderedDict(gptconf.__dict__)
state_dict.update(gptconf_ord_dict)

in_out_dict = {'test_input': test_input, 'test_output': test_output}
state_dict.update(in_out_dict)

state_dict.keys()


data_to_save = state_dict
name_to_save = model_name[:-3] + '_weights.pkl'

with open(name_to_save,'wb') as f:
    pkl.dump(data_to_save, f)



torch.Size([5, 50, 6])
torch.Size([5, 50, 1])


In [24]:
# print(test_output)

In [25]:
# model_chopped = GPT_chopped(gptconf)
# test_input_2 = torch.rand(1,10,8)
# test_output_2 = model_chopped(test_input_2)
# state_dict2 = model_chopped.state_dict()

# gptconf_ord_dict = OrderedDict(gptconf.__dict__)
# state_dict2.update(gptconf_ord_dict)

# in_out_dict = {'test_input': test_input_2, 'test_output': test_output_2}
# state_dict2.update(in_out_dict)

# state_dict2.keys()


# data_to_save = state_dict2
# name_to_save = 'test_chopped_weights.pkl'

# with open(name_to_save,'wb') as f:
#     pkl.dump(data_to_save, f)